# Behaviour Microscope v0.1

**Does an authority cue change how a model handles conflicting UK legal information — and can that change be located and causally tested?**

Interpretability is [Neuronpedia's `interp-engine`](https://www.neuronpedia.org/blog/interp-engine). This notebook is deliberately thin: it sets up the runtime and calls into `src/`.

**Before Run All:** Runtime -> Change runtime type -> a **GPU** with enough memory for 12B (A100 40GB is enough; a T4 is not).

The default model `google/gemma-3-12b-it` is **gated**. Accept the licence on Hugging Face and add your token as a Colab secret named `HF_TOKEN` (key icon, left sidebar). To skip that, set `MODEL_ID = "Qwen/Qwen3-4B"` in the config cell.

## 1. Clone the repository

In [ ]:
import os, sys, pathlib, subprocess

REPO_URL = "https://github.com/ryanmcdonough/behaviour-microscope.git"
REPO_DIR = pathlib.Path("/content/behaviour-microscope")

# Clone if absent; otherwise sync to origin. The old 'clone only if missing' version
# silently ran whatever src/ was already on disk, which is how a stale checkout produces
# a confusing error three cells later.
if not REPO_DIR.exists():
    !git clone --depth 1 $REPO_URL $REPO_DIR
else:
    dirty = subprocess.run(["git", "-C", str(REPO_DIR), "status", "--porcelain"],
                           capture_output=True, text=True).stdout.strip()
    if dirty:
        print("Local changes found -- stashing them before sync:")
        print(dirty)
        !git -C $REPO_DIR stash -u
        print("Recover with: !git -C $REPO_DIR stash pop")
    !git -C $REPO_DIR fetch --depth 1 origin main -q
    !git -C $REPO_DIR reset --hard origin/main -q

os.chdir(REPO_DIR)
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

print("working directory:", pathlib.Path.cwd())
!git -C $REPO_DIR log --oneline -1

## 2. Install

Eager backend only. The vLLM backend needs `enforce_eager=True` to capture at all, which removes most of its speed advantage at this model size — see `RESEARCH.md` section 3.1.

Colab may ask you to restart the session after this cell, if pip needs to change a version of a package Colab already had loaded (numpy or torch, usually). If it does, restart and run from here. `import microscope` itself does not need a restart -- the previous cell puts `src/` on `sys.path` directly.


In [ ]:
# '.[apis]' adds the OpenAI and Anthropic SDKs. Colab ships openai but not anthropic.
%pip install -q -e '.[apis]'
print("installed")

## 3. Verify the GPU

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> GPU, then Run All again."
props = torch.cuda.get_device_properties(0)
print(f"{props.name}  |  {props.total_memory / 1e9:.1f} GB  |  compute {props.major}.{props.minor}")

# T4 (compute 7.5) has no bfloat16 support, so pick the dtype the card can actually run.
DTYPE = "bfloat16" if torch.cuda.is_bf16_supported() else "float16"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("dtype:", DTYPE, " device:", DEVICE)


## 4. Verify interp-engine

The engine version and the `transformers` version are both part of the numerical result, not just dependencies — the engine has no forward pass of its own and hooks `transformers` modules. Both are recorded in the run manifest.

In [ ]:
import interp_engine, transformers
from interp_engine import CAPABILITIES

print("interp-engine", interp_engine.__version__)
print("transformers ", transformers.__version__)
print("vLLM backend available:", interp_engine.vllm_installed())

## 5. Configure the run

In [ ]:
from microscope.experiment import RunConfig, run_all
from microscope.scenarios import ARMS, DEFAULT_CONTRAST

# What this notebook run should do. Exactly one of:
#   "sweep"  -- section 7b runs every model in one session (section 7 is skipped)
#   "single" -- section 7 runs the one model in `cfg` below (section 7b is skipped)
#   "reuse"  -- run nothing; read the newest completed run off disk
MODE = "sweep"

# Model ids, defined here so the sweep and the API section share one source of truth.
# Set OPENAI_MODEL to something your key actually has (the smoke test below lists them).
OPENAI_MODEL = "gpt-5.1"
ANTHROPIC_MODEL = "claude-opus-5"


MODEL_ID = "Qwen/Qwen3-14B"   # ungated alternative: "Qwen/Qwen3-4B"

cfg = RunConfig(
    model_id=MODEL_ID,
    provider="local",           # interp-engine; the only provider that can be patched
    backend="eager",
    dtype=DTYPE,
    extra_load_kwargs={"device": DEVICE},
    n_candidate_layers=4,
    # arms=("floor", "junior_said", "partner_said"),  # narrow it for a cheaper run
    # contrast=DEFAULT_CONTRAST,   # the pair experiments 2-4 patch between
    # limit=5,                     # uncomment for a fast smoke run over 5 scenarios
)

for arm in ARMS:
    cue = arm.cue or "(no assertion)"
    print(f"{arm.name:20s} {cue:40s} {arm.note}")
print()
print("mechanistic contrast:", cfg.contrast, "-- source varies, verb held constant")

### Hugging Face token (gated checkpoints only)

In [ ]:
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded from Colab secrets")
except Exception as exc:
    print(f"No HF_TOKEN secret ({exc}). Fine for an ungated model; gated ones will fail to download.")

## 6. Look at the scenarios before running them

Thirty matched UK legal scenarios, England and Wales. Each generates two prompts that differ only in who is credited with the false proposition.

In [ ]:
from microscope.scenarios import load_scenarios, ARMS
import pandas as pd

scenarios = load_scenarios()
print(len(scenarios), "scenarios")
display(pd.Series([s.area for s in scenarios]).value_counts().rename("scenarios").to_frame())

# The floor arm in full -- this is the base every other arm is built from.
example = scenarios[0]
print(example.prompt("floor"))

# ...and the one line that differs in each asserting arm. Everything else -- the
# evidence, the question, the options and their order -- is byte-identical throughout.
print("\n" + "=" * 78)
print("ADDITIONAL INFORMATION, by arm:\n")
for arm in ARMS:
    if arm.cue:
        print(f"  {arm.name:20s} {arm.cue}")
        print(f"  {'':20s} {example.false_proposition}\n")

## 7. Run the whole experiment

Behaviour, then activation capture, then the bidirectional intervention sweep with its controls. Expect roughly 20-40 minutes on an A100 for 30 scenarios; each phase reports its own timing.


Run as a background thread rather than called directly, because interp-engine's sync facade refuses to run inside an already-active event loop, and Colab's kernel keeps one running for every cell.

In [ ]:
import concurrent.futures
from pathlib import Path

def _newest_completed():
    """Newest run that actually finished -- a summary.json is the completion marker."""
    done = sorted(p for p in Path("results").glob("*Z") if (p / "summary.json").exists())
    if not done:
        raise SystemExit("No completed run on disk. Set MODE to 'single' or 'sweep'.")
    return done[-1]

if MODE == "single":
    # interp-engine's sync facade refuses to run inside an already-running event loop,
    # and Colab's kernel keeps one running for every cell. A worker thread has none.
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
        run_dir = pool.submit(run_all, cfg).result()
elif MODE == "reuse":
    run_dir = _newest_completed()
    print(f"Reusing {run_dir}")
else:
    run_dir = None
    print("MODE is 'sweep' -- skipping the single run; section 7b does the work.")

run_dir

### API keys

Loaded **before** the sweep, because the sweep decides whether to include the API models by checking for these. Loading them later means the API arms are silently dropped.

In [ ]:
# Colab secrets, same mechanism as HF_TOKEN. Grant this notebook access to each one
# (key icon in the left sidebar -> toggle 'Notebook access').
for secret in ("OPENAI_API_KEY", "ANTHROPIC_API_KEY"):
    try:
        from google.colab import userdata
        value = userdata.get(secret)
    except Exception as exc:
        print(f"{secret}: unavailable ({type(exc).__name__}) -- that provider will be skipped")
        continue
    if value:
        os.environ[secret] = value
        print(f"{secret}: loaded ({len(value)} chars)")
    else:
        print(f"{secret}: empty -- that provider will be skipped")

## 7b. The full sweep (all models, one session)

Runs every model one at a time, releasing the card between each, so one `results/` folder holds the whole comparison at one code version.

**Qwen with reasoning ON runs first, deliberately.** It is the only new code path here — the answer arrives after a `<think>` block, so it is parsed from the completion rather than read from the first token. If that is going to fail, it should fail in the first two minutes, not after an hour of GPU time.

Why this set of four:

| run | what it isolates |
| --- | --- |
| Qwen3-14B, thinking **on** | reasoning, holding weights constant |
| Qwen3-14B, thinking **off** | the paired half of that comparison |
| gemma-3-12b-it | a second open family. It has **no** reasoning mode, so there is nothing to toggle |
| gpt-5.1 | frontier reference |

A reasoning run is behavioural-only: with a reasoning block in the way, the answer is no longer at the final prompt position, so patching there would be intervening on the wrong thing. The other two local runs still do the full mechanistic sweep.

In [ ]:
import json
from microscope.experiment import run_sweep, compare_runs

def local(model_id, thinking=False):
    return RunConfig(model_id=model_id, provider="local", backend="eager", dtype=DTYPE,
                     extra_load_kwargs={"device": DEVICE}, n_candidate_layers=4,
                     enable_thinking=thinking)

runs = {}
if MODE != "sweep":
    print(f"MODE is {MODE!r} -- skipping the sweep.")
else:
    sweep = [
        local("Qwen/Qwen3-14B", thinking=True),    # riskiest, so it runs first
        local("Qwen/Qwen3-14B", thinking=False),
        local("google/gemma-3-12b-it"),            # no reasoning mode; the flag is a no-op
    ]
    # API arms join the same sweep, so "sweep" means the whole comparison in one folder.
    for model_id, provider, opts in [(OPENAI_MODEL, "openai", {}),
                                     (ANTHROPIC_MODEL, "anthropic", {"effort": "low"})]:
        if os.environ.get(f"{provider.upper()}_API_KEY"):
            sweep.append(RunConfig(model_id=model_id, provider=provider, provider_options=opts))
        else:
            print(f"SKIPPING {model_id}: no {provider.upper()}_API_KEY -- run the keys cell above.")

    print("\nThis sweep will run:")
    for c in sweep:
        print(f"  - {c.model_id} ({c.provider}"
              + (", thinking" if c.enable_thinking else "") + ")")
    print()

    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
        runs = pool.submit(run_sweep, sweep).result()

    # Sections 8 and 9 examine one run in detail. Prefer a mechanistic one -- it is the
    # only kind with activation and intervention plots to look at.
    for path in runs.values():
        if json.loads((path / "manifest.json").read_text()).get("mechanistic"):
            run_dir = path
            break
    else:
        run_dir = next(iter(runs.values()), None)
    print(f"\nSections 8-9 will examine: {run_dir}")

runs

In [ ]:
# FPAR per arm is the one measure every backend reports -- it needs only the answer letter.
table = compare_runs(runs)
display(table.style.format("{:.0%}").background_gradient(cmap="Reds", vmin=0, vmax=1))

if "Qwen/Qwen3-14B (thinking)" in table.index and "Qwen/Qwen3-14B (no thinking)" in table.index:
    on = table.loc["Qwen/Qwen3-14B (thinking)", "partner_confirmed"]
    off = table.loc["Qwen/Qwen3-14B (no thinking)", "partner_confirmed"]
    print(f"\nSame weights, reasoning toggled -- partner_confirmed: {off:.0%} -> {on:.0%}")
    print("A large drop means the open-vs-frontier gap is substantially about reasoning,")
    print("and 'turn reasoning on' becomes a deployable mitigation. Little change means the")
    print("gap is about something else, and gemma's lack of a reasoning mode is a real limit.")

## 8. Results

In [ ]:
import json

summary = json.loads((run_dir / "summary.json").read_text())
print(json.dumps(summary["behavioural"], indent=2))

### Quality gate

Before reading anything below, check this. It is the same report `run_all()` already wrote to `quality_report.json` -- an automated check on whether *this run's own numbers* are trustworthy, not a check on the code. A model that never engages with the A/B format, a control condition at chance accuracy, or a zero-magnitude patch that isn't actually a no-op each invalidate a different downstream claim; see `src/microscope/quality.py` for what each check is protecting against.

**A `FAIL` here means: do not read the plots as findings.** It usually means the model or prompt format is the wrong choice for this experiment -- itself a useful thing to learn -- not that the pipeline is broken.

In [ ]:
from microscope import quality

report = json.loads((run_dir / "quality_report.json").read_text())
print(quality.format_report(report))

if report["overall"] == "fail":
    print("\n*** At least one check failed. Read the messages above before trusting anything below. ***")

In [ ]:
print("Intervention controls -- read these before reading any effect.")
print("A zero-magnitude patch must not change the output; a magnitude-matched random")
print("direction tells you how much of any effect is just perturbation sensitivity.")
print(json.dumps(summary["intervention_controls"], indent=2))

In [ ]:
from IPython.display import Image, display

for figure in sorted((run_dir / "plots").glob("*.png")):
    print(figure.name)
    display(Image(str(figure)))

## Cross-model replication (OpenAI and Anthropic)

Closed-weights models cannot be captured or patched, so these runs are **behavioural only** -- experiment 1 and nothing else. That is the intended division: the behavioural claim needs breadth across the models legal products actually use, and the mechanism is offered as an explanation of the same phenomenon where we can see inside it.

Two measurement notes that matter when you compare the tables:

- **The Anthropic Messages API exposes no token logprobs.** Claude yields a chosen letter and no probability. That is why the *binary* acceptance rate (FPAR) is the primary cross-model measure everywhere -- it needs only the letter. Pass `samples=k` to estimate a proportion empirically instead, at k times the calls.
- **Reasoning changes the measurement, and is worth testing as a variable.** Set `reasoning_effort` / `effort` and see whether a model that actually works through the statute catches the conflict. That is a finding in its own right.

In [ ]:
# One prompt per provider, before committing to ~210 calls. A wrong model id or an auth
# problem surfaces here in seconds rather than halfway through a run.
from microscope.backends import BackendSpec
from microscope.scenarios import load_scenarios


if os.environ.get("OPENAI_API_KEY"):
    try:
        from openai import OpenAI
        names = sorted(m.id for m in OpenAI().models.list())
        print(f"OpenAI models available ({len(names)}), a sample:")
        print("  " + ", ".join(n for n in names if n.startswith(("gpt", "o")))[:400])
        print(f"\n  '{OPENAI_MODEL}' available: {OPENAI_MODEL in names}")
    except Exception as exc:
        print(f"OpenAI model listing failed: {type(exc).__name__}: {exc}")

probe = load_scenarios()[0].prompt("partner_said")
for kind, model_id, opts in [
    ("openai", OPENAI_MODEL, {}),
    ("anthropic", ANTHROPIC_MODEL, {"effort": "low"}),
]:
    if not os.environ.get(f"{kind.upper()}_API_KEY"):
        print(f"\n{kind}: no key, skipping")
        continue
    try:
        m = BackendSpec(kind=kind, model_id=model_id, options=opts).build().measure(probe)
        print(f"\n{kind} / {model_id}")
        print(f"  letter={m.chosen_letter!r}  parsed={m.parse_ok}  probs={m.probability_source}")
        print(f"  said: {m.generated[:120]!r}")
    except Exception as exc:
        print(f"\n{kind} / {model_id} FAILED: {type(exc).__name__}: {exc}")

In [ ]:
api_runs = {}

if MODE == "sweep":
    # The sweep already ran these; running them again would just spend the API budget twice.
    print("MODE is 'sweep' -- API arms already covered above.")
else:

    api_configs = [
        RunConfig(model_id=OPENAI_MODEL, provider="openai"),
        RunConfig(model_id=ANTHROPIC_MODEL, provider="anthropic",
                  provider_options={"effort": "low"}),
        # Reasoning as a variable -- does working through the statute catch the conflict?
        # RunConfig(model_id=ANTHROPIC_MODEL, provider="anthropic",
        #           provider_options={"effort": "high"}),
    ]

    for api_cfg in api_configs:
        if not os.environ.get(f"{api_cfg.provider.upper()}_API_KEY"):
            print(f"skipping {api_cfg.model_id}: no key")
            continue
        label = f"{api_cfg.model_id} ({api_cfg.provider_options.get('effort', 'default')})"
        try:
            with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
                api_runs[label] = pool.submit(run_all, api_cfg).result()
        except Exception as exc:
            print(f"{label} failed: {type(exc).__name__}: {exc}")

    api_runs

In [ ]:
# FPAR per arm is comparable across every provider -- it needs only the answer letter,
# which is why it is the primary cross-model measure. See RESEARCH.md.
rows = []
for label, path in [(json.loads((run_dir / "manifest.json").read_text())["model"], run_dir),
                    *api_runs.items()]:
    s = json.loads((path / "summary.json").read_text())
    rows.append({"model": label, **s["behavioural"]["fpar_by_arm"]})

table = pd.DataFrame(rows).set_index("model")
display(table.style.format("{:.0%}").background_gradient(cmap="Reds", vmin=0, vmax=1))

## 9. Reading this honestly

Three separate claims, in increasing order of what they would take to support:

1. **Behavioural** — the cue changed the output. Read `fpar` and `authority_deference_delta`. If the delta is near zero and `mcnemar_exact_p` is large, there is no effect here, and the interpretability results below are describing a difference with no behavioural consequence. Say so.
2. **Representational** — the activations differ. Divergence between two conditions is *not* a mechanism. The cue sentences differ lexically as well as in authority, and some of the divergence is that.
3. **Causal** — intervening changed the behaviour. Only credible if the zero control is clean, the random-direction control is small relative to the real patch, and the effect is localised rather than present at every layer. The bidirectional plot is the strongest single piece of evidence: the two curves should move in *opposite* directions at the same layers.

There is no "authority neuron" to find here, and n=30 on one model does not support a claim about language models in general.

## 10. Save the results

Colab discards the filesystem when the runtime ends. `manifest.json` is what makes the run reproducible, and `quality_report.json` is what makes it trustworthy — it pins the checkpoint revision, the engine version and the `transformers` version.

In [ ]:
import shutil

archive = shutil.make_archive(f"/content/{run_dir.name}", "zip", run_dir)
print(archive)

try:
    from google.colab import files
    files.download(archive)
except Exception as exc:
    print(f"Download it from the file browser instead ({exc}).")